# Public self-checks: Student copy

These tests carry **0 marks** and expose only basic cases. No plotting code is loaded or required. Keep this notebook in the same folder as `2026_ID6002W_RL_Programming_Assignment_2.ipynb`, save the assignment notebook, then run all cells here. Do not modify either notebook's cell tags (except for the name of notebook). A 4/4 result does not guarantee full private-test marks.


In [2]:
# Load only setup, provided definitions, and graded functions from the student notebook.
import inspect
import json
import os
import traceback
from pathlib import Path

SUBMISSION_NOTEBOOK = Path("2026_ID6002W_RL_Programming_Assignment_2.ipynb")


def load_submission(path: Path):
    nb = json.loads(path.read_text(encoding="utf-8"))
    namespace = {"__name__": "student_submission"}
    allowed = {"setup", "provided", "graded"}
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        tags = set(cell.get("metadata", {}).get("tags", []))
        if not tags.intersection(allowed) or "figure" in tags:
            continue
        source = "".join(cell.get("source", []))
        exec(compile(source, str(path), "exec"), namespace)
    return namespace


NS = load_submission(SUBMISSION_NOTEBOOK)
np = NS["np"]
torch = NS["torch"]
nn = NS["nn"]

In [ ]:
# Public test Q1: returns, shapes, terminal convention, and reproducibility.
def public_q1():
    np.testing.assert_allclose(
        NS["discounted_returns"]([1.0, -1.0, 2.0], 0.5),
        np.array([1.0, 0.0, 2.0]),
    )
    args = (NS["LoopWorldEnv"](), NS["loopworld_policy"], 1000, 0.95, "first", 17)
    values_a, counts_a = NS["mc_prediction"](*args)
    values_b, counts_b = NS["mc_prediction"](
        NS["LoopWorldEnv"](), NS["loopworld_policy"], 1000, 0.95, "first", 17
    )
    assert values_a.shape == counts_a.shape == (7,)
    assert values_a[0] == values_a[6] == 0.0
    assert counts_a[0] == counts_a[6] == 0
    np.testing.assert_allclose(values_a, values_b)
    np.testing.assert_array_equal(counts_a, counts_b)

In [ ]:
# Public Q2: exact signature, two updates, and seeded action selection.
def public_q2():
    assert list(inspect.signature(NS["epsilon_greedy"]).parameters) == [
        "q_values",
        "epsilon",
        "rng",
    ]
    Q = np.zeros((2, 2))
    Q[0, 0] = 1.0
    Q[1] = [4.0, 3.0]
    assert np.isclose(NS["q_learning_update"](Q, 0, 0, 2.0, 1, False, 0.5, 0.9), 3.3)
    Q = np.zeros((2, 2))
    Q[0, 0] = 1.0
    Q[1] = [4.0, 3.0]
    assert np.isclose(NS["q_learning_update"](Q, 0, 0, 2.0, 1, True, 0.5, 0.9), 1.5)
    rng_a, rng_b = np.random.default_rng(5), np.random.default_rng(5)
    q = np.array([0.0, 2.0, 2.0, -1.0])
    seq_a = [NS["epsilon_greedy"](q, 0.2, rng_a) for _ in range(100)]
    seq_b = [NS["epsilon_greedy"](q, 0.2, rng_b) for _ in range(100)]
    assert seq_a == seq_b

In [ ]:
# Public Q3: decay-before-increment trace recurrence and return structure.
def public_q3():
    trace = np.zeros(3)
    returned = NS["update_accumulating_trace"](trace, 1, 0.9, 0.8)
    assert returned is trace
    NS["update_accumulating_trace"](trace, 2, 0.9, 0.8)
    NS["update_accumulating_trace"](trace, 1, 0.9, 0.8)
    np.testing.assert_allclose(trace, [0.0, 1.5184, 0.72])
    values, diagnostics = NS["td_lambda_prediction"](
        NS["RandomWalk19Env"](), NS["random_walk_policy"], 10, 0.05, 1.0, 0.7, 11
    )
    assert values.shape == (21,)
    assert set(diagnostics) == {"value_history"}
    assert diagnostics["value_history"].shape == (10, 21)
    assert values[0] == values[20] == 0.0

In [ ]:
# Public Q4: normalization, detached target, and prediction API.
def public_q4():
    raw = np.array([[-1.2, -0.07], [0.6, 0.07], [-0.3, 0.0]], dtype=np.float32)
    expected = torch.tensor([[-1.0, -1.0], [1.0, 1.0], [0.0, 0.0]])
    normalized = NS["normalize_states"](raw)
    assert normalized.dtype == torch.float32
    assert normalized.device.type == "cpu"
    torch.testing.assert_close(normalized, expected)

    class ScalarValue(nn.Module):
        def __init__(self):
            super().__init__()
            self.value = nn.Parameter(torch.tensor(4.0))

        def forward(self, states):
            return self.value.expand(states.shape[0])

    model = ScalarValue()
    loss = NS["semi_gradient_td_loss"](
        model,
        torch.zeros((1, 2)),
        torch.tensor([2.0]),
        torch.ones((1, 2)),
        torch.tensor([False]),
        0.9,
    )
    assert loss.ndim == 0
    torch.testing.assert_close(loss, torch.tensor(2.56))
    loss.backward()
    torch.testing.assert_close(model.value.grad, torch.tensor(-3.2))
    prediction = NS["predict_values"](
        NS["ValueNetwork"](), np.array([[-0.3, 0.0]], dtype=np.float32)
    )
    assert isinstance(prediction, np.ndarray) and prediction.shape == (1,)

In [ ]:
# Run one public case per question.
PUBLIC_CASES = [
    ("Q1", public_q1),
    ("Q2", public_q2),
    ("Q3", public_q3),
    ("Q4", public_q4),
]
passed = 0
for name, fn in PUBLIC_CASES:
    try:
        fn()
        passed += 1
        print(f"PASS {name}")
    except Exception as exc:
        print(f"FAIL {name}: {type(exc).__name__}: {exc}")
        traceback.print_exc(limit=1)
print(f"Public tests passed: {passed}/{len(PUBLIC_CASES)}")

PASS Q1
PASS Q2
PASS Q3
PASS Q4
Public tests passed: 4/4
